# SFT Prediction

## Goal
Create a final CSV with three columns:
- **ID**: copied from the input CSV
- **Prompt**: copied from the input CSV
- **Answer**: generated by the fine-tuned adapter model

## Approach
- load the base instruction model: Qwen/Qwen2.5-1.5B-Instruct
- load the trained LoRA adapter on top of the base model
- use 8-bit quantization to reduce GPU memory usage
- process prompts in batches for faster generation
- use resume logic so interrupted runs can continue from the existing output file

In [ ]:
# Import libraries

import os
import math
import time
import pandas as pd
import torch

from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


## 1. Configuration

Here the static parameters for model loading, input/output files, generation, and prompt behavior are defined.


In [ ]:
# Base model and trained adapter path
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = "./sft_out/qwen25_tax_sft_8bit/final_adapter"

# Input CSV and output CSV path
CSV_PATH = "./data/dataset_clean_worktime_version.csv"
OUT_PATH = "./outputs/inference_results_local_8bit_sft_adapter.csv"

# Batch size for parallel generation
BATCH_SIZE = 4

# Maximum number of newly generated tokens per answer
MAX_NEW_TOKENS = 120

# Maximum input length after tokenization
MAX_INPUT_LENGTH = 768

# System prompt defining the response style of the model
SYSTEM_PROMPT = (
    "You are a careful Austrian tax law assistant. "
    "Answer briefly, factually, and clearly in one plain-text paragraph. "
    "Do not invent legal references.")


## 2. GPU Configuration

The notebook checks whether CUDA is available.

This is necessary because the model is intended to run on a GPU.  
Without CUDA, local generation would usually be too slow for practical use.


In [ ]:
# Stop execution if no GPU is available
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available")

# Enable TensorFloat-32 for faster matrix operations on supported NVIDIA GPUs
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

## 3. Load Tokenizer and Model

This step:
1. creates the 8-bit quantization configuration
2. loads the tokenizer
3. sets left padding for batched generation
4. loads the base model
5. loads the trained LoRA adapter on top of the base model
6. switches the model to evaluation mode


In [ ]:
# Use 8-bit quantization to reduce GPU memory usage
quant_config = BitsAndBytesConfig(load_in_8bit=True)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"

# Use EOS as padding token if no pad token is defined
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load the quantized base model
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=quant_config, device_map="auto")

# Load the trained LoRA adapter on top of the base model
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

# Print hardware and model information
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Adapter path:", ADAPTER_PATH)

## 4. Load Input CSV

The input file is read line by line.

Expected format:
- first column: ID
- second column: Prompt

The split is done only at the first comma, so commas inside the prompt are preserved.

In [1]:
# List for storing parsed CSV rows
rows = []

# Read the input CSV file
with open(CSV_PATH, "r", encoding="utf-8-sig") as f:
    next(f, None)  # Skip the header row

    for line_number, line in enumerate(f, start=2):
        line = line.strip()

        # Skip empty lines
        if not line:
            continue

        # Skip malformed lines without a comma separator
        if "," not in line:
            print(f"Skipped line {line_number}")
            continue

        # Split only once: left side = ID, right side = prompt
        id_, prompt = line.split(",", 1)

        rows.append({
            "ID": id_.strip(),
            "Prompt": prompt.strip()
        })

# Convert the parsed rows into a DataFrame
input_df = pd.DataFrame(rows)


NameError: name 'CSV_PATH' is not defined

## 5. Resume Logic

If the output file already exists, the notebook continues from the remaining prompts.

This avoids losing progress when a run was interrupted.


In [ ]:
# Continue from existing output file if available
if os.path.exists(OUT_PATH):
    existing_df = pd.read_csv(OUT_PATH, encoding="utf-8-sig")
    done_ids = set(existing_df["ID"].astype(str))
    print(f"Resume: {len(done_ids)} entries already completed")
else:
    existing_df = pd.DataFrame(columns=["ID", "Prompt", "Answer"])
    done_ids = set()

# Keep only rows that were not processed yet
work_df = input_df[~input_df["ID"].astype(str).isin(done_ids)].copy()
work_df.reset_index(drop=True, inplace=True)

print(f"Remaining items: {len(work_df)} / {len(input_df)}")

# Stop if everything is already finished
if len(work_df) == 0:
    print("Output file already completed")
    raise SystemExit

## 6. Build Chat Input

This helper function converts one prompt into the chat format expected by the Qwen model.


In [ ]:
# Build one chat-formatted input string from a single user prompt
def build_chat_text(prompt: str) -> str:
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

## 7. Batch Prediction Loop

For each batch, the notebook does the following:
1. takes a small group of prompts
2. converts them into chat format
3. tokenizes them together with padding
4. generates answers with the adapter model
5. decodes the generated text
6. appends the results
7. saves progress to the output CSV after each batch

This makes the process faster and safer.


In [ ]:
# Store new prediction results generated in this run
new_results = []

# Compute the total number of batches
num_batches = math.ceil(len(work_df) / BATCH_SIZE)

# Start runtime tracking
start_time = time.time()

for batch_idx in range(num_batches):
    # Select the current batch range
    batch_start = batch_idx * BATCH_SIZE
    batch_end = min(batch_start + BATCH_SIZE, len(work_df))
    batch_df = work_df.iloc[batch_start:batch_end]

    # Extract IDs and prompts for the current batch
    prompts = batch_df["Prompt"].tolist()
    ids = batch_df["ID"].tolist()

    # Convert prompts to chat-formatted input strings
    chat_texts = []
    for prompt in prompts:
        chat_texts.append(build_chat_text(prompt))

    # Tokenize the whole batch together
    inputs = tokenizer(
        chat_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH)

    # Move tokenized inputs to the model device
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Store the real input length from the attention mask
    attention_mask = inputs["attention_mask"]
    input_lengths = attention_mask.sum(dim=1).tolist()

    # Generate answers without computing gradients
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id)

    batch_results = []

    for i in range(len(ids)):
        # With left padding, the generated answer starts after the full padded input length
        padded_input_length = inputs["input_ids"].shape[1]
        generated_tokens = outputs[i][padded_input_length:]

        # Decode and normalize whitespace
        answer = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
        answer = " ".join(answer.split())

        batch_results.append({"ID": ids[i], "Prompt": prompts[i], "Answer": answer})

    # Add batch results to the current run results
    new_results.extend(batch_results)

    # Save intermediate progress after every batch
    temp_df = pd.concat([existing_df, pd.DataFrame(new_results, columns=["ID", "Prompt", "Answer"])], ignore_index=True)
    temp_df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

    # Runtime logging
    processed = batch_end
    elapsed = time.time() - start_time
    per_item = elapsed / processed
    remaining = len(work_df) - processed
    eta_min = (per_item * remaining) / 60

    print(f"Batch {batch_idx + 1}/{num_batches} finished | "
        f"{processed}/{len(work_df)} new prompts | "
        f"ETA approx. {eta_min:.1f} min")

## 8. Final Output

The final result is a CSV file with:
- ID
- Prompt
- Answer


In [ ]:
print(f"Run finished!")